# Missing data and imputation

Real datasets have holes, and how you fill them changes your answers. This notebook distinguishes
the three missingness mechanisms (completely at random, at random, not at random), shows how
complete-case analysis and naive mean imputation each go wrong, then builds principled fixes:
model-based single imputation, multiple imputation with Rubin's rules for honest standard errors, and
a note on how mixle scores partially observed records directly.


## Ignorability: when you can forget the missingness mechanism

Let $R$ indicate which entries are observed and $\phi$ parameterize the missingness mechanism
$p(R\mid x,\phi)$. The data are missing at random (MAR) if $R$ depends only on the observed entries.

Theorem (ignorability). Under MAR and a parameter $\theta$ for the data model that is distinct from
$\phi$, the likelihood factorizes so that the missingness mechanism can be ignored for inference about
$\theta$:
$$p(x_{\text{obs}}, R\mid\theta,\phi) = p(x_{\text{obs}}\mid\theta)\, p(R\mid x_{\text{obs}},\phi),$$
so maximizing the observed-data likelihood $p(x_{\text{obs}}\mid\theta)$ (or imputing from the model) is
valid without modeling $\phi$.

Proof. $p(x_{\text{obs}},R\mid\theta,\phi)=\int p(x_{\text{obs}},x_{\text{mis}}\mid\theta)\,p(R\mid x,\phi)\,dx_{\text{mis}}$.
Under MAR, $p(R\mid x,\phi)=p(R\mid x_{\text{obs}},\phi)$ does not depend on $x_{\text{mis}}$, so it comes
out of the integral, leaving $p(R\mid x_{\text{obs}},\phi)\int p(x_{\text{obs}},x_{\text{mis}}\mid\theta)dx_{\text{mis}}
= p(R\mid x_{\text{obs}},\phi)\,p(x_{\text{obs}}\mid\theta)$. $\quad\blacksquare$

This is the theoretical license for multiple imputation and for mixle's marginalization over missing
fields, both shown in the notebook.

Counterexample (MNAR is not ignorable). If missingness depends on the missing value itself (high earners
decline to report income), the data are missing not at random, the factorization fails, and ignoring the
mechanism biases every estimate. The mechanism must then be modeled jointly, and the answer is sensitive
to assumptions that the data cannot check.

## 1. Three missingness mechanisms

Missing completely at random (MCAR): the holes are independent of everything. Missing at random
(MAR): missingness depends only on observed variables. Missing not at random (MNAR): it depends on the
unobserved value itself. We build a clean dataset and knock out values under a MAR rule where a second
variable controls which y values disappear.


In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.RandomState(0)
n = 1500
x = rng.normal(0, 1, n)
y = 1.0 + 2.0 * x + rng.normal(0, 1, n)                 # true slope 2, intercept 1
p_missing = 1 / (1 + np.exp(-(1.5 * x)))                # MAR: high x -> y more likely missing
miss = rng.random(n) < p_missing
y_obs = y.copy(); y_obs[miss] = np.nan
print('missing fraction of y = %.2f (depends on observed x, so the data are MAR not MCAR)' % miss.mean())

missing fraction of y = 0.51 (depends on observed x, so the data are MAR not MCAR)


## 2. Complete-case analysis biases the mean (but not always the slope)

Dropping rows with any missing value is unbiased only under MCAR. Under our MAR rule the retained rows
are mostly low-x, so the complete-case mean of y is pulled below the truth. A subtle point: the
regression slope of y on x stays roughly unbiased here, because missingness depends only on the
predictor x and not on y given x. Which estimand you care about decides how much the missingness hurts.


In [2]:
from sklearn.linear_model import LinearRegression
cc = ~np.isnan(y_obs)
print('true mean of y           = %.2f' % y.mean())
print('complete-case mean of y  = %.2f  (biased low: high-x high-y rows are mostly missing)' % np.nanmean(y_obs))
b_cc = LinearRegression().fit(x[cc].reshape(-1, 1), y_obs[cc]).coef_[0]
print('complete-case slope      = %.2f  (close to the true 2.0: the slope is robust to MAR on x)' % b_cc)

true mean of y           = 0.92
complete-case mean of y  = -0.12  (biased low: high-x high-y rows are mostly missing)
complete-case slope      = 1.95  (close to the true 2.0: the slope is robust to MAR on x)


## 3. Mean imputation deflates the variance

Replacing missing y by the observed mean keeps the biased complete-case mean and, worse, collapses the
variance: every imputed point sits on one value, so the spread of y is badly understated and any
downstream standard error is too small.


In [3]:
y_mean = y_obs.copy(); y_mean[miss] = np.nanmean(y_obs)
print('mean-imputed mean of y   = %.2f  (same biased value as complete-case)' % y_mean.mean())
print('variance of y: true %.2f, mean-imputed %.2f  (artificially deflated)' % (y.var(), y_mean.var()))

mean-imputed mean of y   = -0.12  (same biased value as complete-case)
variance of y: true 4.72, mean-imputed 1.71  (artificially deflated)


## 4. Model-based and multiple imputation

Conditional (regression) imputation predicts each missing value from the other variables, restoring
the relationship. But a single imputation pretends the filled values are certain. Multiple imputation
draws several completed datasets, analyzes each, and combines them with Rubin's rules so the standard
errors reflect imputation uncertainty.


In [4]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.linear_model import BayesianRidge
from sklearn.impute import IterativeImputer
means = []
for s in range(10):                                     # 10 multiply-imputed datasets
    imp = IterativeImputer(estimator=BayesianRidge(), sample_posterior=True, random_state=s)
    filled = imp.fit_transform(np.column_stack([x, y_obs]))
    means.append(filled[:, 1].mean())
means = np.array(means)
print('multiple-imputation mean of y = %.2f  (true %.2f, recovered)' % (means.mean(), y.mean()))
print('between-imputation SD of the estimate = %.3f  (the uncertainty mean imputation hides)' % means.std(ddof=1))

multiple-imputation mean of y = 0.89  (true 0.92, recovered)
between-imputation SD of the estimate = 0.017  (the uncertainty mean imputation hides)


## 5. mixle models missingness explicitly

mixle treats an optionally present field as a modeled event. An OptionalDistribution emits a
value with probability 1 minus p and is absent (None) with probability p. Because None is a legal
observation, you can fit directly on data that contains holes: the model learns the missingness rate
and the parameters of the present values at the same time, without deleting rows or inventing fills.


In [5]:
from mixle.stats import OptionalEstimator, GaussianEstimator
from mixle.inference.estimation import optimize
truth = rng.normal(5.0, 2.0, 3000)
observed = [None if rng.random() < 0.3 else float(v) for v in truth]   # 30% absent
fit = optimize(observed, OptionalEstimator(GaussianEstimator(), est_prob=True), max_its=1,
               rng=np.random.RandomState(0), print_iter=100)
present = [v for v in observed if v is not None]
print('missing rate: true 0.30, recovered %.2f' % fit.p)
print('present-value mean: data %.2f, recovered %.2f' % (np.mean(present), fit.dist.mu))
print('the model fit straight through the Nones, learning both the hole rate and the value distribution.')

missing rate: true 0.30, recovered 0.31
present-value mean: data 5.02, recovered 5.02
the model fit straight through the Nones, learning both the hole rate and the value distribution.


## References

- Little, R. & Rubin, D. (2019). Statistical Analysis with Missing Data, 3rd ed. Wiley.
- Rubin, D. (1987). Multiple Imputation for Nonresponse in Surveys. Wiley.
- van Buuren, S. & Groothuis-Oudshoorn, K. (2011). mice: Multivariate Imputation by Chained Equations in R. Journal of Statistical Software. https://doi.org/10.18637/jss.v045.i03
- Schafer, J. & Graham, J. (2002). Missing data: our view of the state of the art. Psychological Methods.


## Exercises

1. Make the data MNAR by letting missingness depend on y itself (high earners decline to report), and show that complete-case, mean, and multiple imputation are all biased, because the mechanism is non-ignorable. Discuss what extra modeling MNAR requires.
2. Vary the missing fraction from 5% to 60% and track the bias and standard error of complete-case versus multiple imputation.
3. Implement the EM algorithm for a bivariate Gaussian with missing values directly and compare its maximum-likelihood estimates to the IterativeImputer result.
4. Build a mixle mixture model on a mixed-type dataset with missing fields and confirm that the fitted parameters match a version trained on the fully observed data, demonstrating that marginalization recovers the right answer.
